# 05. ACOS Full Benchmark Evaluation & Interactive Live Inference

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook performs the **final comprehensive evaluation** and live interactive inference for the ACOS benchmark:
- Automatic **Google Drive persistence** at `/content/drive/MyDrive/ACOS/` (saves `05*.ipynb`, benchmark metrics tables, plots, interactive predictions, and markdown reports).
- **Multi-Source Model Checkpoint Detection:** Automatically locates and loads the best fine-tuned checkpoints from Step 1 (`step1_best`) and Step 2 (`step2_best`) across historical Google Drive sessions.
- **Full Benchmark Evaluation:** Evaluates complete end-to-end quadruple extraction across all 15 sub-tasks with micro-metrics and breakdown tables.
- **Interactive Multi-Aspect Inference:** End-to-end two-stage live prediction pipeline on custom raw text reviews.

## 1. Environment, Google Drive Setup & Module Imports
Mounts Google Drive to `/content/drive/MyDrive/ACOS/` and auto-synchronizes notebook & repository assets.

In [ ]:
# 1. Mount Google Drive jika berjalan di Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive berhasil di-mount pada /content/drive")
except Exception:
    print("💻 Berjalan pada lingkungan Lokal / Colab tanpa drive mount.")


Mounted at /content/drive
✅ Google Drive berhasil di-mount pada /content/drive


In [ ]:

# 2. Instalasi dependensi yang dibutuhkan
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3 tqdm

import os
import sys
import re
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 99.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.6 MB/s eta 0:00:00


In [ ]:

# 3. Deteksi dinamis root direktori proyek (Prioritas Utama: Google Drive /content/drive/MyDrive/ACOS)
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
HAS_DRIVE = os.path.exists("/content/drive/MyDrive")

if HAS_DRIVE:
    base_project_dir = "/content/drive/MyDrive/ACOS"
    os.makedirs(base_project_dir, exist_ok=True)
    os.makedirs(os.path.join(base_project_dir, "notebooks"), exist_ok=True)
    save_dir = os.path.join(base_project_dir, "Output")
    os.makedirs(save_dir, exist_ok=True)
    print(f"💾 Mode Google Drive Aktif: {base_project_dir}")
    print(f"📁 Output Sesi akan disimpan di: {base_project_dir}")
elif os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
    print(f"💾 Mode Lokal Aktif (Current Dir): {base_project_dir}")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
    print(f"💾 Mode Lokal Aktif (Parent Dir): {base_project_dir}")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
    print(f"💾 Mode Direktori ACOS: {base_project_dir}")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
    print(f"💾 Mode Colab Ephemeral Aktif: {base_project_dir}")
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
    print(f"💾 Mode Colab /content Aktif: {base_project_dir}")
else:
    base_project_dir = os.path.abspath("ACOS")
    os.makedirs(base_project_dir, exist_ok=True)
    print(f"💾 Inisialisasi folder ACOS: {base_project_dir}")


💾 Mode Google Drive Aktif: /content/drive/MyDrive/ACOS
📁 Output Sesi akan disimpan di: /content/drive/MyDrive/ACOS


In [ ]:

# 4. Auto-clone repositori ACOS jika folder inti belum tersedia di base_project_dir
extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
if not os.path.exists(extract_dir):
    print(f"📥 Repositori belum ditemukan di {base_project_dir}. Mengkloning ACOS dari GitHub...")
    !git clone https://github.com/haisyamalawwab/ACOS.git /tmp/ACOS_clone
    !cp -r /tmp/ACOS_clone/* "{base_project_dir}/"
    !rm -rf /tmp/ACOS_clone
    print("✅ Repositori berhasil disinkronkan.")

notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)


In [ ]:

# 5. Pastikan notebook 05*.ipynb tersimpan di /content/drive/MyDrive/ACOS
def ensure_notebook_saved_to_drive():
    if not HAS_DRIVE:
        return
    cur_nb = "05_ACOS_Evaluation_and_Interactive_Inference.ipynb"
    targets = [
        os.path.join(base_project_dir, cur_nb),
        os.path.join(base_project_dir, "notebooks", cur_nb)
    ]
    sources = [
        cur_nb,
        os.path.join("notebooks", cur_nb),
        os.path.join("/content", cur_nb),
        os.path.join("/content", "ACOS", "notebooks", cur_nb),
        os.path.join("/content", "ACOS", cur_nb),
    ]
    for src in sources:
        if os.path.exists(src):
            src_abs = os.path.abspath(src)
            for tgt in targets:
                tgt_abs = os.path.abspath(tgt)
                if src_abs != tgt_abs:
                    os.makedirs(os.path.dirname(tgt_abs), exist_ok=True)
                    try:
                        shutil.copy2(src_abs, tgt_abs)
                        print(f"💾 Salinan {cur_nb} berhasil disimpan ke: {tgt_abs}")
                    except Exception as e:
                        print(f"⚠️ Gagal menyalin ke {tgt_abs}: {e}")
            break

ensure_notebook_saved_to_drive()


In [ ]:

from modeling import BertForQuadABSA, CategorySentiClassification
from bert_utils.tokenization import BertTokenizer
from run_classifier_dataset_utils import processors, output_modes, convert_examples_to_features2nd
from dataset_utils import read_pair_gold
from eval_metrics import pair_eval, measureQuad, getTextType, measureQuad_imp

# 6. Import colab_utils dengan fallback download
try:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Active PyTorch Device: {device}")
print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")

⚡ Active PyTorch Device: cuda
📂 Base project directory: /content/drive/MyDrive/ACOS
📁 Extract & Model directory: /content/drive/MyDrive/ACOS/Extract-Classify-ACOS


## 2. Session Directory & Model Checkpoint Detection

In [ ]:
DOMAIN = "rest16"   # 'rest16' or 'laptop'

# Pencarian lokasi session folders di Google Drive dan lokal
results_base_candidates = [
    os.path.join(base_project_dir, "results"),
    os.path.join(base_project_dir, "Output", "results"),
    "/content/drive/MyDrive/ACOS/results",
    "/content/drive/MyDrive/ACOS/Output/results"
]
results_base = os.path.join(base_project_dir, "results")
for rb in results_base_candidates:
    if os.path.exists(rb):
        results_base = rb
        break

session_folders = sorted([f for f in os.listdir(results_base) if f.startswith(DOMAIN)]) if os.path.exists(results_base) else []

if session_folders:
    active_session_dir = os.path.join(results_base, session_folders[-1])
    print(f"📂 Menggunakan direktori sesi terbaru: {active_session_dir}")
else:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
    dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)
    active_session_dir = dirs["root"]

plots_dir = os.path.join(active_session_dir, "plots")
csv_dir = os.path.join(active_session_dir, "csv")
logs_dir = os.path.join(active_session_dir, "logs")
checkpoints_dir = os.path.join(active_session_dir, "checkpoints")

bert_fallback = os.path.join(base_project_dir, "bert_base_uncased")
download_bert_pretrained(target_dir=bert_fallback)

# Deteksi komprehensif checkpoint step 1 dan step 2 di seluruh folder Google Drive & lokal
def auto_find_checkpoint(subname):
    # Urutkan prioritas:
    # 1. Folder checkpoints sesi aktif
    # 2. Checkpoints global
    # 3. Semua sesi yang ada di results_base candidates
    candidates = [
        os.path.join(checkpoints_dir, subname),
        os.path.join(base_project_dir, "checkpoints", subname),
        os.path.join(base_project_dir, "Output", "checkpoints", subname),
        f"/content/drive/MyDrive/ACOS/checkpoints/{subname}",
    ]
    found_paths = []
    for c in candidates:
        if os.path.exists(os.path.join(c, "pytorch_model.bin")):
            found_paths.append(c)
            
    for rb in results_base_candidates:
        if os.path.exists(rb):
            for s in sorted(os.listdir(rb), reverse=True):
                p = os.path.join(rb, s, "checkpoints", subname)
                if os.path.exists(os.path.join(p, "pytorch_model.bin")):
                    found_paths.append(p)
                    
    if found_paths:
        # Ambil yang terbaru berdasarkan waktu modifikasi
        found_paths.sort(key=lambda x: os.path.getmtime(os.path.join(x, "pytorch_model.bin")), reverse=True)
        return found_paths[0]
    return bert_fallback

step1_load_dir = auto_find_checkpoint("step1_best")
step2_load_dir = auto_find_checkpoint("step2_best")

print(f"🔹 Step 1 Model: {step1_load_dir}")
print(f"🔹 Step 2 Model: {step2_load_dir}")
md_dir = os.path.join(active_session_dir, "md")
for _d in (plots_dir, csv_dir, logs_dir, md_dir):
    os.makedirs(_d, exist_ok=True)

session_dirs = {"root": active_session_dir, "plots": plots_dir, "csv": csv_dir,
                "logs": logs_dir, "md": md_dir}

rep = MarkdownReport(
    f"05 - Evaluasi Benchmark & Inferensi [{DOMAIN.upper()}]",
    md_dir,
    filename="05_evaluasi_benchmark.md",
    meta={
        "domain": DOMAIN, "session_dir": active_session_dir,
        "step1_model": step1_load_dir, "step2_model": step2_load_dir,
        "checkpoint_step2_terlatih": (step2_load_dir != bert_fallback),
    },
)

if step2_load_dir == bert_fallback:
    print("[PERINGATAN] Checkpoint step 2 hasil training tidak ditemukan.")
    print("             Model memakai bobot BERT mentah, sehingga metrik di bawah")
    print("             TIDAK mencerminkan performa pipeline yang sudah dilatih.")
    rep.text("**Peringatan:** checkpoint step 2 tidak ditemukan; metrik berasal dari bobot BERT mentah.")
print(f"[md] Hasil teks notebook ini ditulis ke: {md_dir}")

📂 Menggunakan direktori sesi terbaru: /content/drive/MyDrive/ACOS/results/rest16_27082026_045140
✅ config.json already cached at /content/drive/MyDrive/ACOS/bert_base_uncased/config.json (0.00 MB)
✅ pytorch_model.bin already cached at /content/drive/MyDrive/ACOS/bert_base_uncased/pytorch_model.bin (420.07 MB)
✅ vocab.txt already cached at /content/drive/MyDrive/ACOS/bert_base_uncased/vocab.txt (0.22 MB)
🔹 Step 1 Model: /content/drive/MyDrive/ACOS/Output/results/rest16_27082026_070817/checkpoints/step1_best
🔹 Step 2 Model: /content/drive/MyDrive/ACOS/Output/results/rest16_27082026_070817/checkpoints/step2_best
[md] Hasil teks notebook ini ditulis ke: /content/drive/MyDrive/ACOS/results/rest16_27082026_045140/md


## 3. Evaluasi Menyeluruh: 15 Kombinasi Sub-Task

`pair_eval` menghitung metrik untuk seluruh 15 kombinasi elemen quadruple (2^4 - 1). Metrik itu hanya ditulis ke logger di `eval_metrics.py`, sehingga notebook menangkapnya lewat `SubtaskMetricCapture` agar bisa ditabelkan.


In [ ]:
processor = processors["categorysenti"]()
label_list = processor.get_labels(DOMAIN)
num_labels = len(label_list[0])
tokenizer = BertTokenizer.from_pretrained(bert_fallback, do_lower_case=True)

tokenized_dir = os.path.join(extract_dir, "tokenized_data")
eval_pair_file, pakai_1st = resolve_eval_pair_file(tokenized_dir, DOMAIN, prefer_1st=True)

eval_examples = pair_examples_from_file(processor, eval_pair_file, set_type="test")

# features_step2 dengan auto-fallback ke gold pair jika tokenisasi gagal
try:
    eval_features = features_step2(eval_examples, label_list, 128, tokenizer, "classification")
except (KeyError, Exception) as _e:
    _gold = os.path.join(tokenized_dir, f"{DOMAIN}_test_pair.tsv")
    if pakai_1st and os.path.exists(_gold):
        print(f"[fallback] Tokenisasi gagal ({type(_e).__name__}: {_e}).")
        print(f"          Beralih ke gold pair: {_gold}")
        eval_pair_file = _gold
        pakai_1st = False
        eval_examples = pair_examples_from_file(processor, eval_pair_file, set_type="test")
        eval_features = features_step2(eval_examples, label_list, 128, tokenizer, "classification")
    else:
        raise

from torch.utils.data import TensorDataset, SequentialSampler, DataLoader
all_input_ids = torch.tensor([f.aspect_input_ids for f in eval_features], dtype=torch.long)
all_input_mask = torch.tensor([f.aspect_input_mask for f in eval_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.aspect_segment_ids for f in eval_features], dtype=torch.long)
all_candidate_aspect = torch.tensor([f.candidate_aspect for f in eval_features], dtype=torch.long)
all_candidate_opinion = torch.tensor([f.candidate_opinion for f in eval_features], dtype=torch.long)
all_label_id = torch.tensor([f.label_id for f in eval_features], dtype=torch.float)
all_tokens_len = torch.tensor([f.tokens_len for f in eval_features], dtype=torch.long)

eval_data = TensorDataset(all_tokens_len, all_input_ids, all_input_mask, all_segment_ids,
                          all_candidate_aspect, all_candidate_opinion, all_label_id)
eval_dataloader = DataLoader(eval_data, sampler=SequentialSampler(eval_data), batch_size=16)

class ArgsProxy:
    def __init__(self):
        self.bert_model = bert_fallback
        self.do_lower_case = True

proxy_args = ArgsProxy()
gold_pair_file = os.path.join(tokenized_dir, f"{DOMAIN}_test_pair.tsv")
with open(gold_pair_file, "r", encoding="utf-8") as f:
    eval_gold = read_pair_gold(f.readlines(), proxy_args)

model_step2 = CategorySentiClassification.from_pretrained(step2_load_dir, num_labels=num_labels)
model_step2.to(device)
model_step2.eval()

class ArgsHelper:
    def __init__(self):
        self.output_dir = logs_dir
        self.max_seq_length = 128

eval_args = ArgsHelper()

import logging
logger = logging.getLogger("Eval_Subtasks")

# pair_eval hanya me-log metrik 15 sub-task; capture agar bisa ditabelkan.
with SubtaskMetricCapture(logger) as cap:
    final_res = pair_eval("eval", eval_args, logger, tokenizer, model_step2, eval_dataloader,
                          eval_gold, label_list, device, "categorysenti", eval_type="test")

subtask_metrics = cap.to_dict()
df_subtasks = cap.to_frame()

df_konteks = pd.DataFrame([{
    "Domain": DOMAIN,
    "Kelas_Label": num_labels,
    "File_Kandidat": os.path.basename(eval_pair_file),
    "Sumber_Kandidat": "prediksi step 1" if pakai_1st else "gold pair",
    "Jumlah_Kandidat": len(eval_examples),
    "Jumlah_Gold_Pair": len(eval_gold[0]),
    "Checkpoint_Step2": step2_load_dir,
}])
rep.section("1. Konteks evaluasi")
export_step_table(df_konteks, name="eval_01_konteks", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Konteks Evaluasi ({DOMAIN.upper()})")
rep.table(df_konteks, caption="Konteks evaluasi")

print(f"\nMetrik quadruple keseluruhan:")
for k, v in final_res.items():
    print(f"  {k}: {v*100:.2f}%")
print(f"Sub-task tertangkap: {len(subtask_metrics)}")


## 4. Tabel Metrik & Ekspor CSV/Markdown

Semua angka di bawah berasal dari evaluasi sesi ini, bukan nilai yang ditulis manual di dalam notebook.


In [ ]:
# Metrik di bawah berasal dari keluaran pair_eval pada sesi ini, bukan angka
# yang ditulis manual. Bila metrik sub-task tidak tertangkap, notebook melaporkan
# kekosongan itu apa adanya alih-alih menampilkan angka pengganti.

rep.section("2. Metrik keseluruhan quadruple")
df_overall = pd.DataFrame([{
    "Metrik": k, "Nilai": v, "Persen": round(v * 100, 2),
} for k, v in final_res.items()])
export_step_table(df_overall, name="eval_02_metrik_quadruple", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Metrik Ekstraksi Quadruple ({DOMAIN.upper()})")
rep.table(df_overall, caption="Metrik quadruple keseluruhan")

rep.section("3. Metrik per sub-task")
if df_subtasks.empty:
    msg = ("Metrik sub-task tidak tertangkap dari log pair_eval. "
           "Tabel dan plot sub-task dilewati.")
    print(f"[catatan] {msg}")
    rep.text(msg)
    df_subtasks_pct = pd.DataFrame()
else:
    df_subtasks_pct = df_subtasks.copy()
    for c in ["Precision", "Recall", "Micro_F1"]:
        df_subtasks_pct[c] = (df_subtasks_pct[c] * 100).round(2)

    export_step_table(df_subtasks_pct, name="eval_03_metrik_subtask",
                      csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Metrik per Sub-Task ({DOMAIN.upper()}) - {len(df_subtasks_pct)} kombinasi",
                      notes=("Setiap baris adalah satu kombinasi elemen quadruple "
                             "(category, sentiment, aspect, opinion) yang dievaluasi bersamaan."),
                      max_rows_md=20)
    rep.table(df_subtasks_pct, max_rows=20, caption="Metrik per sub-task")

    p_sub = os.path.join(plots_dir, "05_benchmark_subtasks_f1.png")
    plot_subtask_metrics(df_subtasks, p_sub,
                         title=f"[{DOMAIN.upper()}] Micro-F1 per Sub-Task (hasil evaluasi sesi ini)")
    rep.image(p_sub, "Micro-F1 per sub-task")

    # Agregasi berdasarkan jumlah elemen: makin banyak elemen makin sulit
    df_agg = (df_subtasks.groupby("N_Elements")
              .agg(Jumlah_Subtask=("Subtask", "count"),
                   Micro_F1_Rata2=("Micro_F1", "mean"),
                   Micro_F1_Min=("Micro_F1", "min"),
                   Micro_F1_Maks=("Micro_F1", "max"))
              .reset_index())
    for c in ["Micro_F1_Rata2", "Micro_F1_Min", "Micro_F1_Maks"]:
        df_agg[c] = (df_agg[c] * 100).round(2)

    rep.section("4. Tingkat kesulitan menurut jumlah elemen")
    export_step_table(df_agg, name="eval_04_agregasi_per_jumlah_elemen",
                      csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Micro-F1 Menurut Jumlah Elemen yang Dievaluasi ({DOMAIN.upper()})",
                      notes="Nilai umumnya menurun saat jumlah elemen bertambah, karena semua elemen harus benar bersamaan.")
    rep.table(df_agg, caption="Agregasi per jumlah elemen")

    plt.figure(figsize=(9, 5))
    x = df_agg["N_Elements"].astype(str)
    plt.bar(x, df_agg["Micro_F1_Rata2"], color="#2b5c8f", edgecolor="black", alpha=0.88,
            label="Rata-rata")
    plt.plot(x, df_agg["Micro_F1_Maks"], "o--", color="#2ecc71", label="Maksimum")
    plt.plot(x, df_agg["Micro_F1_Min"], "s--", color="#e74c3c", label="Minimum")
    for xi, v in zip(x, df_agg["Micro_F1_Rata2"]):
        plt.text(xi, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
    plt.title(f"[{DOMAIN.upper()}] Micro-F1 vs Jumlah Elemen Quadruple", fontsize=12, fontweight="bold")
    plt.xlabel("Jumlah elemen yang dievaluasi bersamaan")
    plt.ylabel("Micro-F1 (%)")
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    p_agg = os.path.join(plots_dir, "06_subtask_difficulty_by_element_count.png")
    plt.savefig(p_agg, dpi=300)
    plt.show()
    plt.close()
    rep.image(p_agg, "Micro-F1 menurun seiring bertambahnya jumlah elemen")
    print(f"[plot] {p_agg}")

# Simpan metrik mentah sebagai JSON
metrics_json = os.path.join(logs_dir, "eval_metrics_sesi.json")
with open(metrics_json, "w", encoding="utf-8") as jf:
    json.dump({"overall": final_res, "subtasks": subtask_metrics,
               "sumber_kandidat": "step1" if pakai_1st else "gold"}, jf, indent=2)
print(f"[metrik] JSON metrik sesi: {metrics_json}")


### Tampilkan Plot Hasil Evaluasi


In [ ]:
from IPython.display import Image, display

eval_plots = [
    ("05_benchmark_subtasks_f1.png", "Micro-F1 per sub-task"),
    ("06_subtask_difficulty_by_element_count.png", "Kesulitan menurut jumlah elemen"),
    ("04_step2_training_loss_f1_curve.png", "Kurva training step 2 (bila tersedia)"),
    ("03_step1_training_loss_f1_curve.png", "Kurva training step 1 (bila tersedia)"),
]

rep.section("5. Visualisasi")
for fname, caption in eval_plots:
    path = os.path.join(plots_dir, fname)
    if os.path.exists(path):
        print(f"[plot] {caption}")
        display(Image(path))
        rep.image(path, caption)
    else:
        print(f"[plot] Tidak ditemukan (dilewati): {fname}")


## 5. Inferensi Dua Tahap pada Teks Bebas

Span aspect/opinion diambil dari CRF step 1, lalu setiap pasangan kandidat diklasifikasi kategori dan sentimennya oleh step 2.


In [ ]:
import re as _re

# Inferensi nyata dua tahap memakai bobot yang dimuat di sel sebelumnya.
# Versi sebelumnya memakai pencocokan kata kunci (if "food" in text ...) yang
# menghasilkan quadruple tanpa melibatkan model sama sekali; itu diganti agar
# keluaran benar-benar berasal dari BertForQuadABSA + CategorySentiClassification.

_seq_processor = processors["quad"]()
_seq_label_list = _seq_processor.get_labels(DOMAIN)
_seq_tags = _seq_label_list[1]          # ['[CLS]','O','I-A','B-A','I-O','B-O']
_catsenti_labels = label_list[0]

model_step1_infer = BertForQuadABSA.from_pretrained(step1_load_dir, num_labels=len(_seq_tags))
model_step1_infer.to(device)
model_step1_infer.eval()

STEP1_TERLATIH = step1_load_dir != bert_fallback
STEP2_TERLATIH = step2_load_dir != bert_fallback
if not (STEP1_TERLATIH and STEP2_TERLATIH):
    print("[PERINGATAN] Sebagian model memakai bobot BERT mentah "
          f"(step1 terlatih={STEP1_TERLATIH}, step2 terlatih={STEP2_TERLATIH}).")
    print("             Keluaran inferensi di bawah tidak bermakna sampai kedua")
    print("             checkpoint hasil training tersedia.")


def _spans_dari_tag(tag_ids):
    """Ubah barisan id tag CRF menjadi span aspect dan opinion.

    Pola mengikuti eval_metrics.pred_eval: indeks 3 = B-A, 2 = I-A (regex '32*'),
    indeks 5 = B-O, 4 = I-O (regex '54*'). Offset dikurangi 1 karena [CLS].
    """
    s = "".join(str(t) for t in tag_ids)
    aspects = [(m.start() - 1, m.end() - 1) for m in _re.finditer(r"32*", s)]
    opinions = [(m.start() - 1, m.end() - 1) for m in _re.finditer(r"54*", s)]
    return aspects, opinions


def analyze_review_quadruples(review_text, domain=None, ambang=0.0, tampilkan_proses=True):
    """Ekstraksi quadruple end-to-end dari satu teks ulasan bebas."""
    domain = domain or DOMAIN
    max_len = 128

    tokens = tokenizer.tokenize(review_text.lower())
    tokens = tokens[: max_len - 2]
    input_tokens = ["[CLS]"] + tokens + ["[SEP]"]
    input_ids = tokenizer.convert_tokens_to_ids(input_tokens)
    attn = [1] * len(input_ids)
    seg = [0] * len(input_ids)
    while len(input_ids) < max_len:
        input_ids.append(0)
        attn.append(0)
        seg.append(0)

    t_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    t_attn = torch.tensor([attn], dtype=torch.long).to(device)
    t_seg = torch.tensor([seg], dtype=torch.long).to(device)
    t_dummy = torch.zeros((1, max_len), dtype=torch.long).to(device)
    t_zero = torch.zeros(1, dtype=torch.long).to(device)

    # --- Tahap 1: span aspect/opinion + deteksi implicit ---
    with torch.no_grad():
        out1 = model_step1_infer(
            aspect_input_ids=t_ids, aspect_labels=t_dummy,
            aspect_token_type_ids=t_seg, aspect_attention_mask=t_attn,
            exist_imp_aspect=t_zero, exist_imp_opinion=t_zero,
        )
    _, logits1 = unpack_model_output(out1)
    pred_tags, imp_a_logit, imp_o_logit = logits1
    tag_ids = pred_tags[0]
    imp_aspect = int(imp_a_logit.argmax(-1).item()) == 1
    imp_opinion = int(imp_o_logit.argmax(-1).item()) == 1

    aspects, opinions = _spans_dari_tag(tag_ids)
    if imp_aspect or not aspects:
        aspects = aspects + [(-1, -1)]
    if imp_opinion or not opinions:
        opinions = opinions + [(-1, -1)]

    if tampilkan_proses:
        print(f"\nTeks    : {review_text}")
        print(f"Token   : {len(tokens)} -> {tokens}")
        print(f"Aspect  : {aspects} (implicit={imp_aspect})")
        print(f"Opinion : {opinions} (implicit={imp_opinion})")

    # --- Tahap 2: kategori + sentimen untuk setiap pasangan kandidat ---
    hasil = []
    for (a_st, a_ed) in aspects:
        for (o_st, o_ed) in opinions:
            cand_a = [0] * max_len
            cand_o = [0] * max_len
            if a_st == -1:
                cand_a[0] = 1                      # posisi [CLS] menandai implicit aspect
            else:
                for i in range(a_st + 1, a_ed + 1):
                    if i < max_len:
                        cand_a[i] = 1
            if o_st == -1:
                cand_o[len(tokens) + 1] = 1        # posisi [SEP] menandai implicit opinion
            else:
                for i in range(o_st + 1, o_ed + 1):
                    if i < max_len:
                        cand_o[i] = 1

            with torch.no_grad():
                out2 = model_step2(
                    tokenizer, 0,
                    aspect_input_ids=t_ids,
                    aspect_token_type_ids=t_seg,
                    aspect_attention_mask=t_attn,
                    candidate_aspect=torch.tensor([cand_a], dtype=torch.long).to(device),
                    candidate_opinion=torch.tensor([cand_o], dtype=torch.long).to(device),
                    label_id=torch.zeros((1, num_labels), dtype=torch.float).to(device),
                )
            _, logits2 = unpack_model_output(out2)
            skor = logits2[0][0].detach().cpu().numpy()

            aktif = [i for i, v in enumerate(skor) if v > ambang]
            if not aktif:
                aktif = [int(skor.argmax())]

            asp_txt = "[IMPLICIT]" if a_st == -1 else " ".join(tokens[a_st:a_ed])
            opi_txt = "[IMPLICIT]" if o_st == -1 else " ".join(tokens[o_st:o_ed])

            for idx in aktif:
                lbl = _catsenti_labels[idx]
                kategori, sentimen = lbl.rsplit("#", 1)
                hasil.append({
                    "Aspect": asp_txt,
                    "Aspect_Span": f"{a_st},{a_ed}",
                    "Category": kategori,
                    "Opinion": opi_txt,
                    "Opinion_Span": f"{o_st},{o_ed}",
                    "Sentiment": {"0": "negative", "1": "neutral", "2": "positive"}.get(sentimen, sentimen),
                    "Skor_Logit": round(float(skor[idx]), 4),
                    "Is_Implicit_Aspect": a_st == -1,
                    "Is_Implicit_Opinion": o_st == -1,
                })

    df = pd.DataFrame(hasil)
    if not df.empty:
        df = df.sort_values("Skor_Logit", ascending=False).reset_index(drop=True)
    return df


### Contoh 1: Ulasan Restoran Multi-Aspek


In [ ]:
sample_review_1 = "The sushi was fresh and exquisite, but the service was extremely slow!"
df_out1 = analyze_review_quadruples(sample_review_1, domain=DOMAIN)

rep.section("6. Contoh inferensi 1 (ulasan restoran)")
rep.text(f"Teks: `{sample_review_1}`")
if df_out1.empty:
    print("Tidak ada quadruple yang dihasilkan.")
    rep.text("Model tidak menghasilkan quadruple untuk teks ini.")
else:
    export_step_table(df_out1, name="infer_01_contoh_restoran", csv_dir=csv_dir, md_dir=md_dir,
                      title="Quadruple Hasil Inferensi - Contoh Restoran",
                      notes="Skor_Logit adalah keluaran mentah sebelum sigmoid; ambang default 0.0.",
                      max_rows_md=20)
    rep.table(df_out1, max_rows=20, caption="Quadruple contoh 1")


### Contoh 2: Ulasan Laptop Multi-Aspek

Ruang label mengikuti `DOMAIN` yang aktif. Untuk kategori laptop, jalankan ulang dengan `DOMAIN = 'laptop'` dan checkpoint domain tersebut.


In [ ]:
sample_review_2 = "Decent laptop with great display, but the battery life is terrible."
df_out2 = analyze_review_quadruples(sample_review_2, domain=DOMAIN)

rep.section("7. Contoh inferensi 2 (ulasan laptop)")
rep.text(f"Teks: `{sample_review_2}`")
rep.text(f"Catatan: ruang label yang dipakai adalah domain **{DOMAIN}**. "
         "Untuk kategori laptop, jalankan ulang notebook dengan `DOMAIN = 'laptop'` "
         "dan checkpoint yang dilatih pada domain tersebut.")
if df_out2.empty:
    print("Tidak ada quadruple yang dihasilkan.")
    rep.text("Model tidak menghasilkan quadruple untuk teks ini.")
else:
    export_step_table(df_out2, name="infer_02_contoh_laptop", csv_dir=csv_dir, md_dir=md_dir,
                      title="Quadruple Hasil Inferensi - Contoh Laptop",
                      max_rows_md=20)
    rep.table(df_out2, max_rows=20, caption="Quadruple contoh 2")


## 6. Ringkasan Artefak & Batasan


In [ ]:
def _list_dir(label, path):
    rows = []
    if os.path.isdir(path):
        for f in sorted(os.listdir(path)):
            fp = os.path.join(path, f)
            if os.path.isfile(fp):
                rows.append({"Jenis": label, "Nama": f,
                             "Ukuran_KB": round(os.path.getsize(fp) / 1024, 1)})
    return rows

df_art = pd.DataFrame(
    _list_dir("CSV", csv_dir) + _list_dir("Plot", plots_dir)
    + _list_dir("Markdown", md_dir) + _list_dir("Log", logs_dir)
)

rep.section("8. Artefak sesi")
if not df_art.empty:
    export_step_table(df_art, name="eval_05_daftar_artefak", csv_dir=csv_dir, md_dir=md_dir,
                      title="Daftar Artefak Sesi Evaluasi", max_rows_md=100)
    rep.table(df_art, max_rows=100, caption="Artefak sesi")

rep.section("9. Batasan").text(
    "- Metrik pada laporan ini berasal dari keluaran `pair_eval` pada sesi ini.\n"
    f"- Sumber kandidat evaluasi: {'prediksi step 1' if pakai_1st else 'gold pair'}.\n"
    f"- Checkpoint step 1 terlatih: {step1_load_dir != bert_fallback}.\n"
    f"- Checkpoint step 2 terlatih: {step2_load_dir != bert_fallback}.\n"
    "- Skor pipeline penuh hanya valid bila kedua checkpoint hasil training tersedia "
    "dan kandidat berasal dari prediksi step 1."
)
rep.text(f"Sesi: `{active_session_dir}`")
report_path = rep.save()

print(f"Laporan Markdown evaluasi: {report_path}")
print(f"Direktori sesi: {active_session_dir}")
print(f"Total artefak terdata: {len(df_art)}")

# Sinkronisasi akhir memastikan notebook 05*.ipynb dan seluruh hasil tersimpan di /content/drive/MyDrive/ACOS
ensure_notebook_saved_to_drive()

# Salin ringkasan evaluasi ke root Google Drive jika tersedia
if HAS_DRIVE:
    for art_csv in ["eval_02_metrik_quadruple.csv", "eval_04_ringkasan_benchmark.csv", "eval_05_daftar_artefak.csv"]:
        src_art = os.path.join(csv_dir, art_csv)
        if os.path.exists(src_art):
            try:
                shutil.copy2(src_art, os.path.join(base_project_dir, art_csv))
                print(f"   - Cadangan CSV evaluasi Google Drive: {art_csv}")
            except Exception:
                pass

print("\n" + "="*70)
print("🎉 RINGKASAN PERSISTENSI GOOGLE DRIVE (/content/drive/MyDrive/ACOS):")
if HAS_DRIVE:
    print(f"   ✅ Notebook 05  : /content/drive/MyDrive/ACOS/notebooks/05_ACOS_Evaluation_and_Interactive_Inference.ipynb")
    print(f"   ✅ Notebook 05  : /content/drive/MyDrive/ACOS/05_ACOS_Evaluation_and_Interactive_Inference.ipynb")
    print(f"   ✅ Model Step 1 : {step1_load_dir}")
    print(f"   ✅ Model Step 2 : {step2_load_dir}")
    print(f"   ✅ Direktori Sesi: {active_session_dir}")
    print(f"   ✅ Laporan MD   : {report_path}")
    print(f"   ✅ Total Artefak: {len(df_art)} berkas tersimpan di Google Drive.")
    print("   💾 Seluruh evaluasi benchmark, tabel metrik, dan inferensi tersimpan aman di Google Drive.")
else:
    print("   💻 Berjalan di lingkungan lokal. Seluruh artifact tersimpan di base_project_dir.")
print("="*70)